# AICellID — Blood Cell Classifier (starter)

Fine-tunes a pretrained ResNet18 on a blood cell dataset.

**Run this in Google Colab** (free GPU): Runtime → Change runtime type → GPU.

## Steps
1. Get a dataset. Easiest: PBC dataset (Mendeley) — ~17k labeled WBC images, 8 classes.
2. Organize into `data/train/<class>/*.jpg` and `data/val/<class>/*.jpg`.
3. Run the cells below to fine-tune.
4. Download `cell_classifier.pt` and drop it into `backend/models/` in the repo.

In [ ]:
!pip install torch torchvision pillow

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DATA_DIR = 'data'  # expects data/train/<class>/ and data/val/<class>/
BATCH = 32
EPOCHS = 5
LR = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
val_ds = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
print('classes:', train_ds.classes)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    acc = correct / len(val_ds)
    print(f'epoch {epoch+1}: train_loss={train_loss:.4f}  val_acc={acc:.4f}')

In [ ]:
torch.save({'state_dict': model.state_dict(), 'classes': train_ds.classes}, 'cell_classifier.pt')
print('saved cell_classifier.pt')